# Auditoria reutilizável do corpus RAG

Notebook consolidado para documentar e repetir as rotinas de auditoria usadas no projeto. Ele preserva o princípio de não alterar os arquivos-fonte durante a auditoria e produz saídas verificáveis em CSV/JSON.

Fluxo: inventário -> consistência de metadados -> inspeção textual -> OCR seletivo -> validação por página -> validação do JSONL -> revisão humana das exceções.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-por tesseract-ocr-chi-sim
!pip -q install pymupdf pytesseract pandas pillow beautifulsoup4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Configuração
Ajuste apenas os caminhos. Não altere os arquivos originais.


In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/ragdiretrizes')
CORPUS_DIR = Path('/content/drive/MyDrive/Colab Notebooks/corpus')
AUDIT_DIR = PROJECT_DIR / 'analysis' / 'audit_colab'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_DIR, CORPUS_DIR, AUDIT_DIR, sep='\n')


## 2. Auditoria consolidada do projeto
Usa o módulo versionado `rag_project.audit_corpus`, para que a mesma verificação possa ser repetida no Colab ou localmente.


In [ ]:
%cd /content/drive/MyDrive/ragdiretrizes
!python -m rag_project.audit_corpus --all --report-dir analysis/audit_colab


## 3. Rotina de OCR seletivo usada no projeto
Esta é a rotina estabilizada a partir do OCR seletivo aplicado aos documentos problemáticos. Ela preserva `document_id`, página, idioma do OCR e métricas de qualidade.


In [ ]:
import re, pandas as pd, pymupdf, pytesseract
from PIL import Image
from io import BytesIO

def clean_text(text):
    text = text.replace('\x0c', ' ')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def metrics(text):
    n = len(text)
    if n == 0:
        return 0, 0.0, 'EMPTY'
    ratio = sum(ch.isalnum() for ch in text) / n
    status = 'LOW_TEXT' if n < 40 else ('LOW_QUALITY' if ratio < 0.25 else 'OK')
    return n, round(ratio, 4), status

def ocr_pdf(document_id, pdf_path, lang, output_dir, dpi=220):
    doc = pymupdf.open(pdf_path)
    parts = [f'# OCR — {document_id}', '', f'- source_file: {pdf_path.name}', f'- ocr_language: {lang}', f'- total_pages: {len(doc)}', '']
    rows = []
    for i in range(len(doc)):
        page = doc[i]
        page_no = i + 1
        pix = page.get_pixmap(dpi=dpi, alpha=False)
        image = Image.open(BytesIO(pix.tobytes('png')))
        text = clean_text(pytesseract.image_to_string(image, lang=lang, config='--oem 1 --psm 6'))
        chars, ratio, status = metrics(text)
        parts += [f'## PAGE {page_no}', '', text if text else '[SEM TEXTO OCR]', '']
        rows.append({'document_id': document_id, 'file_name': pdf_path.name, 'page': page_no, 'ocr_language': lang, 'chars': chars, 'alnum_ratio': ratio, 'page_status': status})
        print(f'{document_id} | página {page_no}/{len(doc)} | chars={chars} | status={status}')
    doc.close()
    output_dir.mkdir(parents=True, exist_ok=True)
    out = output_dir / f'{document_id}.md'
    out.write_text('\n'.join(parts), encoding='utf-8')
    return out, rows


## 4. Resumo automático das páginas OCR
As páginas classificadas como `LOW_TEXT`, `LOW_QUALITY` ou `EMPTY` devem ser revisadas visualmente. Página vazia real não deve ser confundida com falha de OCR.


In [ ]:
def save_ocr_audit(all_rows, output_dir):
    df = pd.DataFrame(all_rows)
    pages_csv = output_dir / 'ocr_validation_pages.csv'
    df.to_csv(pages_csv, index=False, encoding='utf-8-sig')
    summary = (df.groupby('document_id').agg(
        total_pages=('page', 'count'),
        pages_ok=('page_status', lambda s: int((s == 'OK').sum())),
        pages_low_text=('page_status', lambda s: int((s == 'LOW_TEXT').sum())),
        pages_low_quality=('page_status', lambda s: int((s == 'LOW_QUALITY').sum())),
        pages_empty=('page_status', lambda s: int((s == 'EMPTY').sum())),
        total_chars=('chars', 'sum')
    ).reset_index())
    summary['percent_pages_ok'] = (summary['pages_ok'] / summary['total_pages'] * 100).round(2)
    summary['ocr_validation_status'] = summary.apply(lambda r: 'READY_FOR_REVIEW' if r['pages_empty'] == 0 and r['percent_pages_ok'] >= 80 else 'REVIEW_REQUIRED', axis=1)
    summary_csv = output_dir / 'ocr_validation_summary.csv'
    summary.to_csv(summary_csv, index=False, encoding='utf-8-sig')
    display(summary)
    display(df[df['page_status'] != 'OK'])
    return pages_csv, summary_csv


## 5. Inspeção visual de uma página problemática
Use esta célula quando uma página for marcada como curta, vazia ou ruidosa. A decisão final deve ser humana e registrada nas notas de validação.


In [ ]:
import matplotlib.pyplot as plt

def show_pdf_page(pdf_path, page_number, dpi=180):
    doc = pymupdf.open(pdf_path)
    page = doc[page_number - 1]
    pix = page.get_pixmap(dpi=dpi, alpha=False)
    image = Image.open(BytesIO(pix.tobytes('png')))
    plt.figure(figsize=(12, 16))
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    doc.close()


## 6. Validação do dataset JSONL
Confere parse JSON, número de registros, documentos únicos, unicidade de `content_id`, duplicatas e textos vazios.


In [ ]:
import json
jsonl = PROJECT_DIR / 'data' / 'conteudos.jsonl'
rows = [json.loads(x) for x in jsonl.read_text(encoding='utf-8').splitlines() if x.strip()]
print('Registros:', len(rows))
print('Documentos únicos:', len(set(r['document_id'] for r in rows)))
print('content_id únicos:', len(set(r['content_id'] for r in rows)))
print('IDs duplicados:', len(rows) - len(set(r['content_id'] for r in rows)))
print('Textos vazios:', sum(not str(r.get('source_text', '')).strip() for r in rows))


## 7. Critério de liberação
Um documento só deve seguir para indexação quando: (1) possui `document_id` único; (2) está corretamente classificado como fonte nacional ou referência internacional; (3) o texto processado existe; (4) páginas problemáticas foram revisadas; (5) exceções foram documentadas; (6) a rastreabilidade documento-página-trecho foi preservada; e (7) o dataset derivado passa na validação estrutural.

A ausência de texto recuperado não deve ser interpretada automaticamente como ausência conceitual no documento.
